# NLI Incremental Dataset Combination Evaluation: Qwen2-1.5B-Instruct

Evaluates **Qwen/Qwen2-1.5B-Instruct** as a **zero-shot base model** across **all 7 possible dataset combinations** to measure the **incremental contribution** of each dataset.

## Experiment Logic

| Scenario | Datasets Evaluated ON | Type |
|---|---|---|
| S1 | SNLI-TR only | Single |
| S2 | MultiNLI-TR only | Single |
| S3 | TrGLUE only | Single |
| S1+S2 | SNLI-TR + MultiNLI-TR | Pair |
| S1+S3 | SNLI-TR + TrGLUE | Pair |
| S2+S3 | MultiNLI-TR + TrGLUE | Pair |
| S1+S2+S3 | All three datasets | Triple |

## Key Question

> **Which dataset, when added to an existing combination, contributes the most to overall performance?**

**Model:** Qwen2-1.5B-Instruct — 1.5B generative LLM, multilingual, instruct-tuned (SFT/DPO). Zero-shot, no fine-tuning.

**Merging strategy:** `concatenate_datasets()` in memory — originals on HuggingFace are **NOT modified**.

**Metrics:** Accuracy, Macro F1, Weighted F1, Per-class F1, Confusion Matrix (CSV + heatmap)

## 1. Install Dependencies

In [3]:
!pip install -q -U transformers datasets accelerate scikit-learn tqdm huggingface_hub[hf_transfer] seaborn matplotlib pandas

## 2. Imports & Environment Setup

In [1]:
import json
import os
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, concatenate_datasets
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    classification_report,
)
from tqdm import tqdm
import warnings
from transformers import pipeline, AutoTokenizer, GenerationConfig
from transformers.utils import logging as hf_logging
warnings.filterwarnings('ignore', category=UserWarning, module='transformers')
hf_logging.set_verbosity_error()

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False
    print('matplotlib/seaborn not available — PNG heatmaps will be skipped.')

# Enable faster HuggingFace downloads
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Device detection: CUDA (Windows/Linux) > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
elif torch.backends.mps.is_available():
    print('Apple Silicon MPS available; using for acceleration.')
else:
    print('No GPU/MPS detected — running on CPU.')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


No GPU/MPS detected — running on CPU.


## 3. Configuration

In [4]:
# Model
MODEL_ID = 'Qwen/Qwen2-1.5B-Instruct'

# Dataset hub repo
REPO_ID = 'yilmazzey/sdp2-nli'
CONFIGS  = ['snli_tr_1_1', 'multinli_tr_1_1', 'trglue_mnli']

# Eval splits per dataset — test/validation ONLY, train is never touched
EVAL_SPLITS = {
    'snli_tr_1_1'     : ['test'],
    'multinli_tr_1_1' : ['validation_matched', 'validation_mismatched'],
    'trglue_mnli'     : ['test_matched', 'test_mismatched'],
}

# All 7 non-empty subsets — singles, pairs, triple
SCENARIOS = {
    'single_snli'       : ['snli_tr_1_1'],
    'single_mnli'       : ['multinli_tr_1_1'],
    'single_trglue'     : ['trglue_mnli'],
    'pair_snli_mnli'    : ['snli_tr_1_1',     'multinli_tr_1_1'],
    'pair_snli_trglue'  : ['snli_tr_1_1',     'trglue_mnli'],
    'pair_mnli_trglue'  : ['multinli_tr_1_1', 'trglue_mnli'],
    'triple_all'        : ['snli_tr_1_1',     'multinli_tr_1_1', 'trglue_mnli'],
}

# Output root
RESULTS_DIR = 'results incremental'

# Evaluation settings
SAMPLE_LIMIT = 200   # None = whole dataset; set e.g. 200 for quick test
BATCH_SIZE   = 16     # lower to 4-8 if memory is low

LABEL_NAMES = ['entailment', 'neutral', 'contradiction']
NUM_LABELS  = 3


## 4. Load All Dataset Configs

Loaded **read-only** from HuggingFace into local cache — originals are **never modified**.
Only `test` / `validation` splits are used; `train` is never touched.

In [5]:
all_datasets = {}
for cfg in CONFIGS:
    print(f'Loading {REPO_ID} :: {cfg} ...')
    all_datasets[cfg] = load_dataset(REPO_ID, cfg)
    print('  available splits:', list(all_datasets[cfg].keys()))
    print('  eval splits used:', EVAL_SPLITS[cfg])
    for sp in EVAL_SPLITS[cfg]:
        if sp in all_datasets[cfg]:
            print(f'    {sp}: {len(all_datasets[cfg][sp])} rows')


Loading yilmazzey/sdp2-nli :: snli_tr_1_1 ...
  available splits: ['train', 'validation', 'test']
  eval splits used: ['test']
    test: 9824 rows
Loading yilmazzey/sdp2-nli :: multinli_tr_1_1 ...
  available splits: ['train', 'validation_matched', 'validation_mismatched']
  eval splits used: ['validation_matched', 'validation_mismatched']
    validation_matched: 9809 rows
    validation_mismatched: 9825 rows
Loading yilmazzey/sdp2-nli :: trglue_mnli ...


  available splits: ['train', 'validation_matched', 'validation_mismatched', 'test_matched', 'test_mismatched']
  eval splits used: ['test_matched', 'test_mismatched']
    test_matched: 9008 rows
    test_mismatched: 9217 rows


## 5. Load Model & Tokenizer

In [6]:
print(f'Loading {MODEL_ID} ...')

# Device selection: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device = 0
    dtype  = torch.bfloat16
elif torch.backends.mps.is_available():
    device = 'mps'
    dtype  = torch.bfloat16
else:
    device = -1
    dtype  = torch.float32

generator = pipeline(
    'text-generation',
    model=MODEL_ID,
    device=device,
    dtype=dtype,
    trust_remote_code=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Clean GenerationConfig — use only max_new_tokens to avoid max_length conflict warnings
pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
gen_config = GenerationConfig(
    max_new_tokens=5,
    do_sample=False,
    pad_token_id=pad_id,
    temperature=None,
    top_p=None,
    top_k=None,
)
# Overwrite the model's baked-in generation config entirely
generator.model.generation_config = gen_config
# Explicitly clear any inherited max_length defaults (e.g., 20)
if hasattr(generator.model.generation_config, 'max_length'):
    generator.model.generation_config.max_length = None
if hasattr(generator.model.config, 'max_length'):
    generator.model.config.max_length = None

print('Model loaded successfully.')


Loading Qwen/Qwen2-1.5B-Instruct ...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded successfully.


## 6. Helper Functions

**Prompt strategy:** Zero-shot chat-template NLI. System message enforces a single-word answer.

| Parsed output | Label index |
|---|---|
| `entailment` | 0 |
| `neutral` | 1 |
| `contradiction` | 2 |

In [7]:
def nli_prompt(premise, hypothesis):
    return [
        {'role': 'system', 'content': 'You are a helpful assistant for natural language inference. '
                                      'Classify the relationship between premise and hypothesis as '
                                      'entailment, neutral, or contradiction. '
                                      'Respond with exactly one word only: entailment, neutral, or contradiction. '
                                      'No explanation, no other text.'},
        {'role': 'user',   'content': f'Premise: {premise}\nHypothesis: {hypothesis}\nLabel:'}
    ]


def parse_label(gen_text, formatted_prompt):
    """Extract predicted NLI label from the generated continuation."""
    continuation = gen_text[len(formatted_prompt):].strip().lower()
    if not continuation:
        return 1  # neutral fallback
    first_word = continuation.split()[0].rstrip('.,!?;:')
    return {'entailment': 0, 'neutral': 1, 'contradiction': 2}.get(first_word, 1)


def run_inference(split_data, n=None):
    """Batched zero-shot inference on a HuggingFace split."""
    n = min(n, len(split_data)) if n is not None else len(split_data)
    premises    = [split_data[i]['premise']    for i in range(n)]
    hypotheses  = [split_data[i]['hypothesis'] for i in range(n)]
    true_labels = [split_data[i]['label']      for i in range(n)]
    y_pred = []
    for start in tqdm(range(0, n, BATCH_SIZE), desc='Inference'):
        end       = min(start + BATCH_SIZE, n)
        prompts   = [nli_prompt(p, h) for p, h in zip(premises[start:end], hypotheses[start:end])]
        formatted = tokenizer.apply_chat_template(prompts, tokenize=False, add_generation_prompt=True)
        out       = generator(
            formatted,
        )
        for i, gen in enumerate(out):
            y_pred.append(parse_label(gen[0]['generated_text'], formatted[i]))
    return true_labels, y_pred


def compute_metrics(y_true, y_pred):
    """Returns full metrics dict + confusion matrix."""
    acc         = float(accuracy_score(y_true, y_pred))
    f1_macro    = float(f1_score(y_true, y_pred, average='macro',    zero_division=0))
    f1_weighted = float(f1_score(y_true, y_pred, average='weighted', zero_division=0))
    f1_per      = f1_score(y_true, y_pred, average=None,             zero_division=0)
    f1_per_cls  = {LABEL_NAMES[i]: float(f1_per[i]) for i in range(NUM_LABELS)}
    cm          = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cr          = classification_report(y_true, y_pred, target_names=LABEL_NAMES,
                                        zero_division=0, output_dict=True)
    return {
        'accuracy'              : acc,
        'f1_macro'              : f1_macro,
        'f1_weighted'           : f1_weighted,
        'f1_per_class'          : f1_per_cls,
        'classification_report' : cr,
    }, cm


def save_confusion(cm, out_dir, name):
    """Save confusion matrix as CSV + optional PNG heatmap."""
    global HAS_PLOT
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    csv_path = out_dir / f'confusion_{name}.csv'
    np.savetxt(csv_path, cm, fmt='%d', delimiter=',',
               header=','.join(LABEL_NAMES), comments='')
    print(f'  Confusion CSV  -> {csv_path}')
    if HAS_PLOT:
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d',
                    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
        ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(name)
        plt.tight_layout()
        png_path = out_dir / f'confusion_{name}.png'
        plt.savefig(png_path); plt.close()
        print(f'  Confusion PNG  -> {png_path}')


def collect_splits(cfg_list):
    """Returns list of (cfg_name, split_name, split_data) tuples."""
    result = []
    for cfg in cfg_list:
        for sp in EVAL_SPLITS[cfg]:
            if sp in all_datasets[cfg]:
                result.append((cfg, sp, all_datasets[cfg][sp]))
            else:
                print(f'  Warning: {cfg}/{sp} not found, skipping.')
    return result


def run_scenario(scenario_name, cfg_list):
    """
    Evaluate the model on all eval splits of every dataset in cfg_list.
    Saves per-split metrics, a merged combined score, confusion matrices,
    and a metrics.json under results incremental/<scenario_name>/.

    Args:
        scenario_name : folder name, e.g. 'pair_snli_mnli'
        cfg_list      : list of dataset config names to evaluate on
    Returns:
        dict with full results
    """
    out_dir = Path(RESULTS_DIR) / scenario_name
    out_dir.mkdir(parents=True, exist_ok=True)

    print('\n' + '='*65)
    print(f'SCENARIO : {scenario_name}')
    print(f'Datasets : {cfg_list}')
    print('='*65)

    splits         = collect_splits(cfg_list)
    per_split_m    = {}
    all_y_true, all_y_pred = [], []

    for cfg, sp, split_data in splits:
        key = f'{cfg}__{sp}'
        print(f'\n  Evaluating {cfg} / {sp}  ({len(split_data)} rows) ...')
        y_true, y_pred = run_inference(split_data, n=SAMPLE_LIMIT)
        print('    True dist:', dict(Counter(y_true)))
        print('    Pred dist:', dict(Counter(y_pred)))
        metrics, cm = compute_metrics(y_true, y_pred)
        per_split_m[key] = metrics
        all_y_true.extend(y_true)
        all_y_pred.extend(y_pred)
        print(f"    acc={metrics['accuracy']:.4f}  f1_macro={metrics['f1_macro']:.4f}  f1_weighted={metrics['f1_weighted']:.4f}")
        save_confusion(cm, out_dir, key)

    print(f'\n  Combined over {len(splits)} split(s) ...')
    combined_m, combined_cm = compute_metrics(all_y_true, all_y_pred)
    print(f"  COMBINED acc={combined_m['accuracy']:.4f}  f1_macro={combined_m['f1_macro']:.4f}  f1_weighted={combined_m['f1_weighted']:.4f}")
    save_confusion(combined_cm, out_dir, f'{scenario_name}_combined')

    result = {
        'model'     : MODEL_ID,
        'scenario'  : scenario_name,
        'datasets'  : cfg_list,
        'n_splits'  : len(splits),
        'per_split' : per_split_m,
        'combined'  : combined_m,
    }
    with open(out_dir / 'metrics.json', 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)
    print(f"\n  Saved -> {out_dir / 'metrics.json'}")
    return result


## 7. Single Dataset Scenarios

Baseline: each dataset evaluated **individually**.
These scores are the reference points for measuring incremental gain.

In [8]:
result_single_snli = run_scenario(
    scenario_name = 'single_snli',
    cfg_list      = ['snli_tr_1_1']
)



SCENARIO : single_snli
Datasets : ['snli_tr_1_1']

  Evaluating snli_tr_1_1 / test  (9824 rows) ...


Inference: 100%|██████████| 13/13 [10:53<00:00, 50.24s/it]


    True dist: {1: 65, 0: 69, 2: 66}
    Pred dist: {1: 111, 0: 77, 2: 12}
    acc=0.4850  f1_macro=0.4357  f1_weighted=0.4395
  Confusion CSV  -> results incremental\single_snli\confusion_snli_tr_1_1__test.csv
  Confusion PNG  -> results incremental\single_snli\confusion_snli_tr_1_1__test.png

  Combined over 1 split(s) ...
  COMBINED acc=0.4850  f1_macro=0.4357  f1_weighted=0.4395
  Confusion CSV  -> results incremental\single_snli\confusion_single_snli_combined.csv
  Confusion PNG  -> results incremental\single_snli\confusion_single_snli_combined.png

  Saved -> results incremental\single_snli\metrics.json


In [9]:
result_single_mnli = run_scenario(
    scenario_name = 'single_mnli',
    cfg_list      = ['multinli_tr_1_1']
)



SCENARIO : single_mnli
Datasets : ['multinli_tr_1_1']

  Evaluating multinli_tr_1_1 / validation_matched  (9809 rows) ...


Inference: 100%|██████████| 13/13 [12:43<00:00, 58.75s/it]


    True dist: {1: 59, 2: 59, 0: 82}
    Pred dist: {1: 90, 2: 23, 0: 87}
    acc=0.4950  f1_macro=0.4546  f1_weighted=0.4758
  Confusion CSV  -> results incremental\single_mnli\confusion_multinli_tr_1_1__validation_matched.csv
  Confusion PNG  -> results incremental\single_mnli\confusion_multinli_tr_1_1__validation_matched.png

  Evaluating multinli_tr_1_1 / validation_mismatched  (9825 rows) ...


Inference: 100%|██████████| 13/13 [15:02<00:00, 69.46s/it]


    True dist: {2: 66, 0: 65, 1: 69}
    Pred dist: {2: 34, 0: 78, 1: 88}
    acc=0.4600  f1_macro=0.4433  f1_weighted=0.4420
  Confusion CSV  -> results incremental\single_mnli\confusion_multinli_tr_1_1__validation_mismatched.csv
  Confusion PNG  -> results incremental\single_mnli\confusion_multinli_tr_1_1__validation_mismatched.png

  Combined over 2 split(s) ...
  COMBINED acc=0.4775  f1_macro=0.4491  f1_weighted=0.4592
  Confusion CSV  -> results incremental\single_mnli\confusion_single_mnli_combined.csv
  Confusion PNG  -> results incremental\single_mnli\confusion_single_mnli_combined.png

  Saved -> results incremental\single_mnli\metrics.json


In [10]:
result_single_trglue = run_scenario(
    scenario_name = 'single_trglue',
    cfg_list      = ['trglue_mnli']
)



SCENARIO : single_trglue
Datasets : ['trglue_mnli']

  Evaluating trglue_mnli / test_matched  (9008 rows) ...


Inference: 100%|██████████| 13/13 [16:45<00:00, 77.38s/it]


    True dist: {1: 74, 2: 59, 0: 67}
    Pred dist: {1: 75, 0: 82, 2: 43}
    acc=0.6850  f1_macro=0.6769  f1_weighted=0.6794
  Confusion CSV  -> results incremental\single_trglue\confusion_trglue_mnli__test_matched.csv
  Confusion PNG  -> results incremental\single_trglue\confusion_trglue_mnli__test_matched.png

  Evaluating trglue_mnli / test_mismatched  (9217 rows) ...


Inference: 100%|██████████| 13/13 [16:37<00:00, 76.73s/it]


    True dist: {1: 65, 0: 73, 2: 62}
    Pred dist: {1: 78, 0: 83, 2: 39}
    acc=0.7100  f1_macro=0.6976  f1_weighted=0.7033
  Confusion CSV  -> results incremental\single_trglue\confusion_trglue_mnli__test_mismatched.csv
  Confusion PNG  -> results incremental\single_trglue\confusion_trglue_mnli__test_mismatched.png

  Combined over 2 split(s) ...
  COMBINED acc=0.6975  f1_macro=0.6875  f1_weighted=0.6911
  Confusion CSV  -> results incremental\single_trglue\confusion_single_trglue_combined.csv
  Confusion PNG  -> results incremental\single_trglue\confusion_single_trglue_combined.png

  Saved -> results incremental\single_trglue\metrics.json


## 8. Pair Dataset Scenarios

All three possible two-dataset combinations.
Comparing each pair score to its two single scores reveals the **incremental gain** from combining.

In [11]:
result_pair_snli_mnli = run_scenario(
    scenario_name = 'pair_snli_mnli',
    cfg_list      = ['snli_tr_1_1', 'multinli_tr_1_1']
)



SCENARIO : pair_snli_mnli
Datasets : ['snli_tr_1_1', 'multinli_tr_1_1']

  Evaluating snli_tr_1_1 / test  (9824 rows) ...


Inference: 100%|██████████| 13/13 [14:26<00:00, 66.63s/it]


    True dist: {1: 65, 0: 69, 2: 66}
    Pred dist: {1: 114, 0: 72, 2: 14}
    acc=0.4750  f1_macro=0.4359  f1_weighted=0.4387
  Confusion CSV  -> results incremental\pair_snli_mnli\confusion_snli_tr_1_1__test.csv
  Confusion PNG  -> results incremental\pair_snli_mnli\confusion_snli_tr_1_1__test.png

  Evaluating multinli_tr_1_1 / validation_matched  (9809 rows) ...


Inference: 100%|██████████| 13/13 [16:11<00:00, 74.71s/it]


    True dist: {1: 59, 2: 59, 0: 82}
    Pred dist: {1: 83, 0: 81, 2: 36}
    acc=0.4950  f1_macro=0.4734  f1_weighted=0.4909
  Confusion CSV  -> results incremental\pair_snli_mnli\confusion_multinli_tr_1_1__validation_matched.csv
  Confusion PNG  -> results incremental\pair_snli_mnli\confusion_multinli_tr_1_1__validation_matched.png

  Evaluating multinli_tr_1_1 / validation_mismatched  (9825 rows) ...


Inference: 100%|██████████| 13/13 [17:01<00:00, 78.58s/it]


    True dist: {2: 66, 0: 65, 1: 69}
    Pred dist: {1: 92, 2: 28, 0: 80}
    acc=0.4750  f1_macro=0.4607  f1_weighted=0.4596
  Confusion CSV  -> results incremental\pair_snli_mnli\confusion_multinli_tr_1_1__validation_mismatched.csv
  Confusion PNG  -> results incremental\pair_snli_mnli\confusion_multinli_tr_1_1__validation_mismatched.png

  Combined over 3 split(s) ...
  COMBINED acc=0.4817  f1_macro=0.4589  f1_weighted=0.4657
  Confusion CSV  -> results incremental\pair_snli_mnli\confusion_pair_snli_mnli_combined.csv
  Confusion PNG  -> results incremental\pair_snli_mnli\confusion_pair_snli_mnli_combined.png

  Saved -> results incremental\pair_snli_mnli\metrics.json


In [12]:
result_pair_snli_trglue = run_scenario(
    scenario_name = 'pair_snli_trglue',
    cfg_list      = ['snli_tr_1_1', 'trglue_mnli']
)



SCENARIO : pair_snli_trglue
Datasets : ['snli_tr_1_1', 'trglue_mnli']

  Evaluating snli_tr_1_1 / test  (9824 rows) ...


Inference: 100%|██████████| 13/13 [14:17<00:00, 65.94s/it]


    True dist: {1: 65, 0: 69, 2: 66}
    Pred dist: {1: 116, 0: 71, 2: 13}
    acc=0.4700  f1_macro=0.4265  f1_weighted=0.4303
  Confusion CSV  -> results incremental\pair_snli_trglue\confusion_snli_tr_1_1__test.csv
  Confusion PNG  -> results incremental\pair_snli_trglue\confusion_snli_tr_1_1__test.png

  Evaluating trglue_mnli / test_matched  (9008 rows) ...


Inference: 100%|██████████| 13/13 [17:16<00:00, 79.70s/it]


    True dist: {1: 74, 2: 59, 0: 67}
    Pred dist: {1: 74, 0: 85, 2: 41}
    acc=0.7050  f1_macro=0.6973  f1_weighted=0.6989
  Confusion CSV  -> results incremental\pair_snli_trglue\confusion_trglue_mnli__test_matched.csv
  Confusion PNG  -> results incremental\pair_snli_trglue\confusion_trglue_mnli__test_matched.png

  Evaluating trglue_mnli / test_mismatched  (9217 rows) ...


Inference: 100%|██████████| 13/13 [16:12<00:00, 74.84s/it]


    True dist: {1: 65, 0: 73, 2: 62}
    Pred dist: {1: 75, 0: 83, 2: 42}
    acc=0.7300  f1_macro=0.7210  f1_weighted=0.7250
  Confusion CSV  -> results incremental\pair_snli_trglue\confusion_trglue_mnli__test_mismatched.csv
  Confusion PNG  -> results incremental\pair_snli_trglue\confusion_trglue_mnli__test_mismatched.png

  Combined over 3 split(s) ...
  COMBINED acc=0.6350  f1_macro=0.6201  f1_weighted=0.6244
  Confusion CSV  -> results incremental\pair_snli_trglue\confusion_pair_snli_trglue_combined.csv
  Confusion PNG  -> results incremental\pair_snli_trglue\confusion_pair_snli_trglue_combined.png

  Saved -> results incremental\pair_snli_trglue\metrics.json


In [13]:
result_pair_mnli_trglue = run_scenario(
    scenario_name = 'pair_mnli_trglue',
    cfg_list      = ['multinli_tr_1_1', 'trglue_mnli']
)



SCENARIO : pair_mnli_trglue
Datasets : ['multinli_tr_1_1', 'trglue_mnli']

  Evaluating multinli_tr_1_1 / validation_matched  (9809 rows) ...


Inference: 100%|██████████| 13/13 [17:35<00:00, 81.18s/it]


    True dist: {1: 59, 2: 59, 0: 82}
    Pred dist: {1: 94, 0: 84, 2: 22}
    acc=0.5000  f1_macro=0.4560  f1_weighted=0.4784
  Confusion CSV  -> results incremental\pair_mnli_trglue\confusion_multinli_tr_1_1__validation_matched.csv
  Confusion PNG  -> results incremental\pair_mnli_trglue\confusion_multinli_tr_1_1__validation_matched.png

  Evaluating multinli_tr_1_1 / validation_mismatched  (9825 rows) ...


Inference: 100%|██████████| 13/13 [16:37<00:00, 76.77s/it]


    True dist: {2: 66, 0: 65, 1: 69}
    Pred dist: {2: 34, 0: 80, 1: 86}
    acc=0.5100  f1_macro=0.4976  f1_weighted=0.4963
  Confusion CSV  -> results incremental\pair_mnli_trglue\confusion_multinli_tr_1_1__validation_mismatched.csv
  Confusion PNG  -> results incremental\pair_mnli_trglue\confusion_multinli_tr_1_1__validation_mismatched.png

  Evaluating trglue_mnli / test_matched  (9008 rows) ...


Inference: 100%|██████████| 13/13 [16:37<00:00, 76.72s/it]


    True dist: {1: 74, 2: 59, 0: 67}
    Pred dist: {1: 77, 0: 86, 2: 37}
    acc=0.6800  f1_macro=0.6680  f1_weighted=0.6716
  Confusion CSV  -> results incremental\pair_mnli_trglue\confusion_trglue_mnli__test_matched.csv
  Confusion PNG  -> results incremental\pair_mnli_trglue\confusion_trglue_mnli__test_matched.png

  Evaluating trglue_mnli / test_mismatched  (9217 rows) ...


Inference: 100%|██████████| 13/13 [15:53<00:00, 73.32s/it]


    True dist: {1: 65, 0: 73, 2: 62}
    Pred dist: {1: 77, 0: 84, 2: 39}
    acc=0.6850  f1_macro=0.6683  f1_weighted=0.6754
  Confusion CSV  -> results incremental\pair_mnli_trglue\confusion_trglue_mnli__test_mismatched.csv
  Confusion PNG  -> results incremental\pair_mnli_trglue\confusion_trglue_mnli__test_mismatched.png

  Combined over 4 split(s) ...
  COMBINED acc=0.5938  f1_macro=0.5746  f1_weighted=0.5813
  Confusion CSV  -> results incremental\pair_mnli_trglue\confusion_pair_mnli_trglue_combined.csv
  Confusion PNG  -> results incremental\pair_mnli_trglue\confusion_pair_mnli_trglue_combined.png

  Saved -> results incremental\pair_mnli_trglue\metrics.json


## 9. Triple Dataset Scenario (All Combined)

All three datasets merged together — the upper-bound of dataset coverage.

In [14]:
result_triple_all = run_scenario(
    scenario_name = 'triple_all',
    cfg_list      = ['snli_tr_1_1', 'multinli_tr_1_1', 'trglue_mnli']
)



SCENARIO : triple_all
Datasets : ['snli_tr_1_1', 'multinli_tr_1_1', 'trglue_mnli']

  Evaluating snli_tr_1_1 / test  (9824 rows) ...


Inference: 100%|██████████| 13/13 [16:03<00:00, 74.10s/it]


    True dist: {1: 65, 0: 69, 2: 66}
    Pred dist: {1: 111, 0: 79, 2: 10}
    acc=0.4650  f1_macro=0.4265  f1_weighted=0.4299
  Confusion CSV  -> results incremental\triple_all\confusion_snli_tr_1_1__test.csv
  Confusion PNG  -> results incremental\triple_all\confusion_snli_tr_1_1__test.png

  Evaluating multinli_tr_1_1 / validation_matched  (9809 rows) ...


Inference: 100%|██████████| 13/13 [15:12<00:00, 70.22s/it]


    True dist: {1: 59, 2: 59, 0: 82}
    Pred dist: {1: 90, 2: 28, 0: 82}
    acc=0.5100  f1_macro=0.4865  f1_weighted=0.5021
  Confusion CSV  -> results incremental\triple_all\confusion_multinli_tr_1_1__validation_matched.csv
  Confusion PNG  -> results incremental\triple_all\confusion_multinli_tr_1_1__validation_matched.png

  Evaluating multinli_tr_1_1 / validation_mismatched  (9825 rows) ...


...

    True dist: {2: 66, 0: 65, 1: 69}
    Pred dist: {2: 35, 1: 86, 0: 79}
    acc=0.4650  f1_macro=0.4529  f1_weighted=0.4520
  Confusion CSV  -> results incremental\triple_all\confusion_multinli_tr_1_1__validation_mismatched.csv
  Confusion PNG  -> results incremental\triple_all\confusion_multinli_tr_1_1__validation_mismatched.png

  Evaluating trglue_mnli / test_matched  (9008 rows) ...


Inference: 100%|██████████| 13/13 [14:16<00:00, 65.87s/it]


    True dist: {1: 74, 2: 59, 0: 67}
    Pred dist: {1: 73, 0: 85, 2: 42}
    acc=0.6900  f1_macro=0.6833  f1_weighted=0.6842
  Confusion CSV  -> results incremental\triple_all\confusion_trglue_mnli__test_matched.csv
  Confusion PNG  -> results incremental\triple_all\confusion_trglue_mnli__test_matched.png

  Evaluating trglue_mnli / test_mismatched  (9217 rows) ...


Inference: 100%|██████████| 13/13 [12:02<00:00, 55.60s/it]


    True dist: {1: 65, 0: 73, 2: 62}
    Pred dist: {1: 79, 0: 84, 2: 37}
    acc=0.7350  f1_macro=0.7213  f1_weighted=0.7264
  Confusion CSV  -> results incremental\triple_all\confusion_trglue_mnli__test_mismatched.csv
  Confusion PNG  -> results incremental\triple_all\confusion_trglue_mnli__test_mismatched.png

  Combined over 5 split(s) ...
  COMBINED acc=0.5730  f1_macro=0.5567  f1_weighted=0.5621
  Confusion CSV  -> results incremental\triple_all\confusion_triple_all_combined.csv
  Confusion PNG  -> results incremental\triple_all\confusion_triple_all_combined.png

  Saved -> results incremental\triple_all\metrics.json


## 10. Final Summary — Incremental Contribution Analysis

Builds three outputs:
1. **All 7 scenarios ranked** by F1 Macro
2. **Incremental gain table** — pair vs. average of its two singles; triple vs. each pair
3. **Key findings** — best scenario, best pair gain, which dataset adds the most

Saves master `results incremental/metrics.json`.

In [15]:
all_results = {
    'single_snli'       : result_single_snli,
    'single_mnli'       : result_single_mnli,
    'single_trglue'     : result_single_trglue,
    'pair_snli_mnli'    : result_pair_snli_mnli,
    'pair_snli_trglue'  : result_pair_snli_trglue,
    'pair_mnli_trglue'  : result_pair_mnli_trglue,
    'triple_all'        : result_triple_all,
}

# ── Table 1: all scenarios ranked by F1 Macro ───────────────────────────
rows = []
for name, res in all_results.items():
    m = res['combined']
    n_ds = len(res['datasets'])
    rows.append({
        'Scenario'         : name,
        'Type'             : 'single' if n_ds == 1 else ('pair' if n_ds == 2 else 'triple'),
        'Datasets'         : ' + '.join(res['datasets']),
        'Accuracy'         : round(m['accuracy'],    4),
        'F1 Macro'         : round(m['f1_macro'],    4),
        'F1 Weighted'      : round(m['f1_weighted'], 4),
        'F1 Entailment'    : round(m['f1_per_class']['entailment'],   4),
        'F1 Neutral'       : round(m['f1_per_class']['neutral'],      4),
        'F1 Contradiction' : round(m['f1_per_class']['contradiction'], 4),
    })

summary_df = pd.DataFrame(rows).set_index('Scenario').sort_values('F1 Macro', ascending=False)
print('=' * 80)
print(f'ALL SCENARIOS RANKED BY F1 MACRO  |  Model: {MODEL_ID}')
print('=' * 80)
print(summary_df.to_string())

# ── Table 2: incremental gain ────────────────────────────────────────────
pair_map = {
    'pair_snli_mnli'   : ('single_snli',   'single_mnli',   'trglue_mnli'),
    'pair_snli_trglue' : ('single_snli',   'single_trglue', 'multinli_tr_1_1'),
    'pair_mnli_trglue' : ('single_mnli',   'single_trglue', 'snli_tr_1_1'),
}
gain_rows = []
triple_f1 = all_results['triple_all']['combined']['f1_macro']
for pair_name, (s1, s2, missing) in pair_map.items():
    pair_f1   = all_results[pair_name]['combined']['f1_macro']
    avg_s_f1  = (all_results[s1]['combined']['f1_macro'] +
                 all_results[s2]['combined']['f1_macro']) / 2
    gain_rows.append({
        'Pair'                 : pair_name,
        'Missing Dataset'      : missing,
        'Avg Single F1'        : round(avg_s_f1,            4),
        'Pair F1 Macro'        : round(pair_f1,             4),
        'Gain vs Avg Single'   : round(pair_f1 - avg_s_f1,  4),
        'Triple F1 Macro'      : round(triple_f1,           4),
        'Gain Pair -> Triple'  : round(triple_f1 - pair_f1, 4),
    })

gain_df = pd.DataFrame(gain_rows).set_index('Pair')
print('\n' + '=' * 80)
print('INCREMENTAL GAIN ANALYSIS')
print('=' * 80)
print(gain_df.to_string())

# ── Key findings ────────────────────────────────────────────────────────
print('\n' + '=' * 80)
print('KEY FINDINGS')
print('=' * 80)
print(f'  Best  scenario        : {summary_df.index[0]}  (F1 Macro = {summary_df.iloc[0]["F1 Macro"]:.4f})')
print(f'  Worst scenario        : {summary_df.index[-1]}  (F1 Macro = {summary_df.iloc[-1]["F1 Macro"]:.4f})')
best_pair = gain_df['Gain vs Avg Single'].idxmax()
print(f'  Best pair gain        : {best_pair}  (+{gain_df.loc[best_pair, "Gain vs Avg Single"]:.4f} over avg single)')
best_boost = gain_df['Gain Pair -> Triple'].idxmax()
print(f'  Pair gaining most from 3rd dataset : {best_boost}  (missing: {gain_df.loc[best_boost, "Missing Dataset"]})')

# ── Save master metrics.json ─────────────────────────────────────────────
master = Path(RESULTS_DIR) / 'metrics.json'
master.parent.mkdir(parents=True, exist_ok=True)
with open(master, 'w', encoding='utf-8') as f:
    json.dump({
        'model'      : MODEL_ID,
        'experiment' : 'incremental_dataset_combination_evaluation',
        'scenarios'  : all_results,
    }, f, indent=2, ensure_ascii=False)
print(f'\nMaster metrics saved -> {master}')


ALL SCENARIOS RANKED BY F1 MACRO  |  Model: Qwen/Qwen2-1.5B-Instruct
                    Type                                     Datasets  Accuracy  F1 Macro  F1 Weighted  F1 Entailment  F1 Neutral  F1 Contradiction
Scenario                                                                                                                                           
single_trglue     single                                  trglue_mnli    0.6975    0.6875       0.6911         0.7803      0.6712            0.6108
pair_snli_trglue    pair                    snli_tr_1_1 + trglue_mnli    0.6350    0.6201       0.6244         0.7545      0.5970            0.5088
pair_mnli_trglue    pair                multinli_tr_1_1 + trglue_mnli    0.5938    0.5746       0.5813         0.7118      0.5624            0.4497
triple_all        triple  snli_tr_1_1 + multinli_tr_1_1 + trglue_mnli    0.5730    0.5567       0.5621         0.6876      0.5344            0.4483
pair_snli_mnli      pair                snl